# T1.L2 — Класифікація математичних моделей

Мета: на одному об'єкті порівняти детерміновану, стохастичну, дискретну динамічну та Monte Carlo моделі.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

LESSON_DIR = Path.cwd()
if LESSON_DIR.name == "notebooks":
    LESSON_DIR = LESSON_DIR.parent
SRC = LESSON_DIR / "src"
sys.path.insert(0, str(SRC))

from model import (
    ResourceModelParams,
    deterministic_stock,
    deterministic_exhaustion_time,
    stochastic_stock_path,
    monte_carlo_exhaustion_times,
    summarize_exhaustion,
)


## 1. Детермінована модель

\[
S(t)=\max(0,S_0-vt)
\]

За `S0=120`, `v=6` модель дає один точний прогноз у межах прийнятих припущень.

In [ ]:
S0 = 120
v = 6
horizon = 21
time = np.arange(horizon + 1)

det = deterministic_stock(time, S0, v)
print("Deterministic exhaustion time:", deterministic_exhaustion_time(S0, v))

plt.figure(figsize=(8,4))
plt.plot(time, det)
plt.title("Deterministic model")
plt.xlabel("Step")
plt.ylabel("Remaining stock")
plt.grid(True, alpha=.3)
plt.show()


## 2. Стохастична + дискретна динамічна модель

\[
C_k \sim N(\mu,\sigma^2), \qquad
S_{k+1}=\max(0,S_k-C_k)
\]

Один запуск — лише одна реалізація процесу.

In [ ]:
params = ResourceModelParams(
    initial_stock=120,
    mean_consumption=6,
    std_consumption=1.5,
    horizon=21,
)

s7, c7 = stochastic_stock_path(params, seed=7)
s21, c21 = stochastic_stock_path(params, seed=21)

plt.figure(figsize=(8,4))
plt.plot(time, det, label="deterministic")
plt.plot(time, s7, label="stochastic seed=7")
plt.plot(time, s21, label="stochastic seed=21")
plt.title("One object — different model realizations")
plt.xlabel("Step")
plt.ylabel("Remaining stock")
plt.legend()
plt.grid(True, alpha=.3)
plt.show()


## 3. Reproducibility

Однаковий `seed` має відтворити ту саму траєкторію.

In [ ]:
s7_repeat, _ = stochastic_stock_path(params, seed=7)
print("Same path:", np.allclose(s7, s7_repeat))


## 4. Monte Carlo

Повторюємо стохастичну модель багато разів і аналізуємо не одну траєкторію, а розподіл результатів.

In [ ]:
times = monte_carlo_exhaustion_times(params, n_runs=3000, seed=2026)
summary = summarize_exhaustion(times)
summary


In [ ]:
finite = times[np.isfinite(times)]
plt.figure(figsize=(8,4))
plt.hist(finite, bins=np.arange(finite.min()-0.5, finite.max()+1.5, 1))
plt.title("Monte Carlo: distribution of exhaustion step")
plt.xlabel("Exhaustion step")
plt.ylabel("Runs")
plt.show()


## 5. Сценарії

Порівняємо зміну середнього рівня споживання та зміну варіативності.

In [ ]:
scenarios = pd.read_csv(LESSON_DIR / "data" / "scenarios.csv")
rows = []
for _, row in scenarios.iterrows():
    p = ResourceModelParams(
        initial_stock=row.initial_stock,
        mean_consumption=row.mean_consumption,
        std_consumption=row.std_consumption,
        horizon=int(row.horizon),
    )
    t = monte_carlo_exhaustion_times(p, n_runs=3000, seed=2026)
    s = summarize_exhaustion(t)
    rows.append({
        "scenario": row.scenario,
        "deterministic_T": deterministic_exhaustion_time(p.initial_stock, p.mean_consumption),
        **s,
    })
pd.DataFrame(rows)


## Висновок

Класифікація моделі має практичний зміст: вона визначає структуру припущень, тип результату та допустимий рівень висновків.

**Research transfer:** оберіть один процес власного дисертаційного дослідження і сформулюйте для нього детерміновану та стохастичну версії.